In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def is_circular_image(img_np, threshold=20):
    try:
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        h, w = gray.shape
        corners = [
            gray[0:20, 0:20],
            gray[0:20, w-20:w],
            gray[h-20:h, 0:20],
            gray[h-20:h, w-20:w]
        ]
        dark_corners = [np.mean(c) < threshold for c in corners]
        return sum(dark_corners) >= 3
    except:
        return False

def is_mostly_black(img_np, threshold=0.85):
    try:
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        return (np.sum(gray < 15) / gray.size) > threshold
    except:
        return True


def find_circle_robust(img_np):
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape

    for blur_k in [(9, 9), (15, 15), (21, 21)]:
        blurred = cv2.GaussianBlur(gray, blur_k, 2)
        circles = cv2.HoughCircles(
            blurred, cv2.HOUGH_GRADIENT,
            dp=1, minDist=min(h, w) // 2,
            param1=50, param2=25,
            minRadius=int(min(h, w) * 0.25),
            maxRadius=int(min(h, w) * 0.65)
        )
        if circles is not None:
            return tuple(map(int, np.around(circles[0][0])))

    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        largest = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest)
        perimeter = cv2.arcLength(largest, True)
        if perimeter > 0 and (4 * np.pi * area) / (perimeter ** 2) > 0.5:
            (cx, cy), r = cv2.minEnclosingCircle(largest)
            return int(cx), int(cy), int(r)

    return w // 2, h // 2, int(min(h, w) * 0.48)



def crop_circle_content(img_np):
    cx, cy, r = find_circle_robust(img_np)
    h, w = img_np.shape[:2]
    side = int(0.95 * r / np.sqrt(2))
    x1, x2 = max(cx - side, 0), min(cx + side, w)
    y1, y2 = max(cy - side, 0), min(cy + side, h)
    cropped = img_np[y1:y2, x1:x2]
    return cropped if (cropped.size > 0 and not is_mostly_black(cropped)) else None


def remove_black_edges(img_np):
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    coords = np.column_stack(np.where(gray > 20))
    if coords.size == 0:
        return img_np
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    cropped = img_np[y_min:y_max+1, x_min:x_max+1]
    return cropped if cropped.size > 0 else img_np


def crop_to_square(img_np):
    h, w = img_np.shape[:2]
    size = min(h, w)
    sx, sy = (w - size) // 2, (h - size) // 2
    sq = img_np[sy:sy+size, sx:sx+size]
    return sq if sq.size > 0 else img_np


def process_image_safe(img_path):
    pil_img = Image.open(img_path).convert("RGB")
    img_np = np.array(pil_img)
    original_np = img_np.copy()

    if is_circular_image(img_np):
        result = crop_circle_content(img_np)
        img_np = result if (result is not None and not is_mostly_black(result)) else remove_black_edges(original_np)
    else:
        img_np = remove_black_edges(img_np)

    if img_np is None or is_mostly_black(img_np):
        img_np = original_np

    img_np = crop_to_square(img_np)
    return Image.fromarray(img_np).resize((224, 224), Image.LANCZOS)



def validate_on_folder(folder_path, max_images=7000, cols=5):
    """
    Loads up to max_images from folder_path,
    shows Original vs Processed side by side in a grid.
    """
    import os

    exts = ('.jpg', '.jpeg', '.png', '.bmp')
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(exts)][:max_images]

    if not files:
        print("No images found in folder.")
        return

    rows = (len(files) + cols - 1) // cols
    fig, axes = plt.subplots(rows * 2, cols, figsize=(cols * 3, rows * 6))
    fig.suptitle("Top row = Original   |   Bottom row = Processed", fontsize=13, y=1.01)


    axes = np.array(axes).reshape(rows * 2, cols)

    for idx, fname in enumerate(files):
        col = idx % cols
        row_orig = (idx // cols) * 2
        row_proc = row_orig + 1

        img_path = os.path.join(folder_path, fname)

        try:
            original = np.array(Image.open(img_path).convert("RGB"))
            processed = np.array(process_image_safe(img_path))

            axes[row_orig][col].imshow(original)
            axes[row_orig][col].set_title(fname[:15], fontsize=7)
            axes[row_orig][col].axis("off")

            axes[row_proc][col].imshow(processed)
            axes[row_proc][col].set_title("processed", fontsize=7)
            axes[row_proc][col].axis("off")

        except Exception as e:
            axes[row_orig][col].set_title(f"ERR: {fname[:10]}", fontsize=7)
            axes[row_orig][col].axis("off")
            axes[row_proc][col].axis("off")
            print(f"Failed: {fname} → {e}")


    for idx in range(len(files), rows * cols):
        col = idx % cols
        row_orig = (idx // cols) * 2
        axes[row_orig][col].axis("off")
        axes[row_orig + 1][col].axis("off")

    plt.tight_layout()
    plt.savefig("validation_output.jpg", dpi=150, bbox_inches="tight")
    plt.show()
    print(" Saved → validation_output.jpg")


def validate_single(img_path):
    """
    Deep debug on one image — shows 4 panels:
    Original | Circle detected | Cropped | Final 224x224
    """
    pil_img = Image.open(img_path).convert("RGB")
    img_np  = np.array(pil_img)


    cx, cy, r = find_circle_robust(img_np)
    debug_img = img_np.copy()
    cv2.circle(debug_img, (cx, cy), r,  (0, 255, 0),   3)
    cv2.circle(debug_img, (cx, cy), 5,  (255, 0,   0), -1)
    side = int(0.95 * r / np.sqrt(2))
    cv2.rectangle(debug_img,
                  (cx - side, cy - side),
                  (cx + side, cy + side),
                  (255, 165, 0), 3)


    result_crop = crop_circle_content(img_np)
    cropped_show = result_crop if result_crop is not None else img_np


    final = np.array(process_image_safe(img_path))

    is_circ = is_circular_image(img_np)

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    titles = [
        f"Original\n{img_np.shape[1]}x{img_np.shape[0]}",
        f"Circle Detection\ncenter=({cx},{cy}) r={r}\ncircular={is_circ}",
        f"Cropped Content\n{cropped_show.shape[1]}x{cropped_show.shape[0]}",
        "Final 224x224"
    ]
    imgs = [img_np, debug_img, cropped_show, final]

    for ax, im, title in zip(axes, imgs, titles):
        ax.imshow(im)
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig("debug_single.jpg", dpi=150, bbox_inches="tight")
    plt.show()
    print(" Saved → debug_single.jpg")

In [ ]:
import os
from PIL import Image
import numpy as np
from tqdm import tqdm

def process_and_save(input_folder, output_folder):
    """
    - Reads every image from input_folder
    - If black edges detected → saves cropped version
    - If no black edges → saves original resized to 224x224
    - Total images in == total images out, always
    - Never overwrites already processed images
    """
    os.makedirs(output_folder, exist_ok=True)

    exts = ('.jpg', '.jpeg', '.png', '.bmp')
    files = [f for f in os.listdir(input_folder) if f.lower().endswith(exts)]

    if not files:
        print("No images found in input folder.")
        return

    ok_count   = 0
    skip_count = 0
    fail_count = 0

    for fname in tqdm(files, desc="Processing"):
        img_path  = os.path.join(input_folder, fname)
        save_path = os.path.join(output_folder, fname)


        if os.path.exists(save_path):
            skip_count += 1
            continue

        try:
            original_pil = Image.open(img_path).convert("RGB")
            original_np  = np.array(original_pil)

            processed    = process_image_safe(img_path)
            processed_np = np.array(processed)


            orig_resized = np.array(original_pil.resize((224, 224)))
            diff = np.mean(np.abs(orig_resized.astype(int) - processed_np.astype(int)))

            if diff > 3.0:

                processed.save(save_path)
                ok_count += 1
            else:

                original_pil.resize((224, 224), Image.LANCZOS).save(save_path)
                ok_count += 1

        except Exception as e:
            print(f"Error: {fname} → {e}")

            try:
                Image.open(img_path).convert("RGB").resize((224, 224), Image.LANCZOS).save(save_path)
                fail_count += 1
            except:
                print(f"  Could not save fallback for {fname}")




process_and_save(
    input_folder  = "/content/drive/MyDrive/data_22_april/dataset to drive/VASC",
    output_folder = "/content/drive/MyDrive/data_22_april/cropped/VASC"
)

In [ ]:
import os

DATASET_PATH = "/content/drive/MyDrive/data_22_april/PREPROCESSING/RESIZED ALL"

for cls in os.listdir(DATASET_PATH):
    class_path = os.path.join(DATASET_PATH, cls)

    if os.path.isdir(class_path):
        num_images = len(os.listdir(class_path))
        print(f"{cls}: {num_images} images")

AKIEC: 655 images
BCC: 410 images
BKL: 875 images
DF: 184 images
MEL: 881 images
NV: 5220 images
VASC: 202 images
